# 17 — Recalibrate + re-tune the blend against the 5-variant ensemble

**What notebook 16 found (2026-09-10)**: the equal-weight ensemble of all 5
same-preprocessing training-protocol variants (rung3 + familybias + lrsched
+ augment + classweight, 25 repeat-level OOF arrays) beats the rung-3-only
5-array ensemble by -0.0105 log loss (0.4022 vs. 0.4127), moving with AUROC
too (+0.0069) -- a real, coherent win. **Adopted as the production ensemble
composition.**

Notebook 15's calibration (T_cnn=1.00, T_baseline=0.50, w=0.70) was fit
against the *rung-3-only* CNN signal. Since the "CNN side" of the blend is
now a richer 5-variant ensemble with a different ECE (0.0360 vs. rung-3's
0.0316) and different AUROC, those parameters need to be **re-fit**, not
carried over -- this notebook is notebook 15's exact method, applied to the
new composition.

**Per-repeat "CNN" signal, redefined**: for each of the 5 seeds, this
notebook's `cnn_oof_repeats[i]` is now the average of that seed's 5 variant
OOF arrays (rung3, familybias, lrsched, augment, classweight) -- i.e. each
"repeat" going into the LOFO calibration is itself already a 5-variant
ensemble for that seed. The classical-baseline side is unchanged (reuses
`baseline_oof_seed{42..46}.npy`, already computed and cached by notebook 15
-- no need to recompute).

**Data handling**: loads real row-level labels and OOF prediction arrays,
so per the AI-assistant data rule (`README.md`) this is **[RUN ME]** — run
it yourself, share back only the printed aggregate numbers. CPU-only, no
GPU, no volume cache -- seconds to low tens of seconds for the grid
search.

In [ ]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays
# (all 5 non-denoise variants + the per-seed classical baseline OOF
# notebook 15 already cached). CPU-only, no GPU, no volume cache --
# self-contained, does not assume any earlier cell/notebook ran in this
# kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
y_true = np.array(labels)

repeat_seeds = list(range(config.SEED, config.SEED + 5))
VARIANT_PREFIXES = ["rung3", "rung4_familybias", "rung4_lrsched", "rung4_augment", "rung4_classweight"]

# per-seed 5-variant-ensembled "CNN" signal -- this replaces notebook 15's
# plain rung3_oof_seed{s}.npy with the richer, adopted ensemble.
cnn_oof_repeats = []
for s in repeat_seeds:
    variant_arrays = [np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy") for prefix in VARIANT_PREFIXES]
    cnn_oof_repeats.append(np.mean(variant_arrays, axis=0))

# classical baseline OOF, per seed -- reuse notebook 15's cache if present,
# else recompute the same way it did (should already exist on disk).
baseline_feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
baseline_feat_df = baseline_feat_df.set_index(config.UID_COLUMN).loc[uids].reset_index()
baseline_X = baseline_feat_df[["abs_asym", "striatal_ratio"]].to_numpy()
baseline_y = baseline_feat_df[config.TARGET_COLUMN].to_numpy()
baseline_family = baseline_feat_df["inplane_family"].to_numpy()
assert np.array_equal(baseline_y, y_true), "baseline_features.csv row order must match labels_df merge"

baseline_oof_repeats = []
for seed in repeat_seeds:
    cache_path = config.DATA_PROCESSED / f"baseline_oof_seed{seed}.npy"
    if cache_path.exists():
        baseline_oof_repeats.append(np.load(cache_path))
        continue
    baseline_oof = np.zeros(len(uids))
    folds = evaluate.make_folds(baseline_y, baseline_family, n_splits=config.N_FOLDS, random_state=seed)
    for train_idx, test_idx in folds:
        pipeline = model.build_combat_baseline()
        pipeline.fit(baseline_X[train_idx], baseline_y[train_idx], baseline_family[train_idx])
        baseline_oof[test_idx] = pipeline.predict_proba(baseline_X[test_idx], baseline_family[test_idx])[:, 1]
    np.save(cache_path, baseline_oof)
    baseline_oof_repeats.append(baseline_oof)

print(f"{len(repeat_seeds)} repeats ready: 5-variant-ensembled CNN + seed-matched baseline OOF.")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# LOFO-validated joint grid search over (T_cnn, T_baseline, w) in logit
# space -- identical method to notebook 15, applied to the 5-variant CNN
# signal instead of rung3-only.
T_GRID = np.arange(0.5, 2.01, 0.1)
W_GRID = np.arange(0.0, 1.01, 0.05)
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def calibrated_blend(cnn_p, baseline_p, t_cnn, t_baseline, w):
    combined_logit = w * (to_logit(cnn_p) / t_cnn) + (1 - w) * (to_logit(baseline_p) / t_baseline)
    return 1.0 / (1.0 + np.exp(-combined_logit))


def fast_log_loss(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


cnn_logits = [to_logit(oof) for oof in cnn_oof_repeats]
baseline_logits = [to_logit(oof) for oof in baseline_oof_repeats]

honest_scores = []
selected_params = []
for held_out_i in range(len(repeat_seeds)):
    selection = [i for i in range(len(repeat_seeds)) if i != held_out_i]
    best = None
    for t_cnn in T_GRID:
        cnn_scaled = [cnn_logits[i] / t_cnn for i in selection]
        for t_base in T_GRID:
            base_scaled = [baseline_logits[i] / t_base for i in selection]
            for w in W_GRID:
                mean_ll = np.mean([
                    fast_log_loss(y_true, 1.0 / (1.0 + np.exp(-(w * cs + (1 - w) * bs))))
                    for cs, bs in zip(cnn_scaled, base_scaled)
                ])
                if best is None or mean_ll < best[0]:
                    best = (mean_ll, float(t_cnn), float(t_base), float(w))
    _, t_cnn, t_base, w = best
    held_out_probs = calibrated_blend(cnn_oof_repeats[held_out_i], baseline_oof_repeats[held_out_i], t_cnn, t_base, w)
    held_out_score = evaluate.log_loss_score(y_true, held_out_probs)
    honest_scores.append(held_out_score)
    selected_params.append((t_cnn, t_base, w))
    print(f"  held-out repeat {held_out_i} (seed={repeat_seeds[held_out_i]}): "
          f"selected T_cnn={t_cnn:.2f}, T_baseline={t_base:.2f}, w={w:.2f} on the other 4, "
          f"scored {held_out_score:.4f} on this one")

honest_scores = np.array(honest_scores)
print(f"\nLOFO calibrated-blend (5-variant ensemble): mean={honest_scores.mean():.4f}, "
      f"sd={honest_scores.std(ddof=1):.4f}")
print("for comparison -- notebook 15's rung3-only LOFO calibrated-blend: mean=0.4006 sd=0.0096")
print("for comparison -- notebook 16's uncalibrated 5-variant ensemble (pooled): log loss=0.4022")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Final candidate params (mean of the 5 LOFO-selected triples). NOTE: as in
# notebook 15, this pooled-ensemble application has mild optimism relative
# to the honest per-repeat LOFO mean above -- trust the LOFO mean/sd for
# the go/no-go decision, use this only as the actual recipe to implement.
final_t_cnn = float(np.mean([p[0] for p in selected_params]))
final_t_baseline = float(np.mean([p[1] for p in selected_params]))
final_w = float(np.mean([p[2] for p in selected_params]))
print(f"final params (mean of 5 LOFO-selected triples): "
      f"T_cnn={final_t_cnn:.2f}, T_baseline={final_t_baseline:.2f}, w={final_w:.2f}")

cnn_ensemble_oof = np.mean(cnn_oof_repeats, axis=0)
baseline_ensemble_oof = np.mean(baseline_oof_repeats, axis=0)
candidate_oof = calibrated_blend(cnn_ensemble_oof, baseline_ensemble_oof, final_t_cnn, final_t_baseline, final_w)

candidate_scores = evaluate.combined_score(y_true, candidate_oof)
print(f"\npooled candidate (calibrated, re-tuned blend, 5-variant ensemble): "
      f"log loss={candidate_scores['log_loss']:.4f}  AUROC={candidate_scores['auroc']:.4f}  "
      f"ECE={candidate_scores['ece']:.4f}")
print("real leaderboard (first submission): log loss=0.4648 AUROC=0.8796")

**What we're looking for:** with the 5-variant ensemble adopted (notebook
16), does re-fitting calibration + blend weight against it (instead of
carrying over notebook 15's rung3-only params) give a further, honest LOFO
improvement -- and where do T_cnn/T_baseline/w land this time?

**What we found:** all 5 LOFO folds independently selected the **identical**
triple -- T_cnn=0.50, T_baseline=1.00, w=0.45 -- perfect unanimity, even
stronger than notebook 15's 4/5. Held-out scores: [0.3884, 0.3783, 0.3718,
0.3742, 0.3820], mean=**0.3789**, sd=0.0066 (tighter than notebook 15's
0.0096 too). Vs. notebook 15's rung3-only LOFO calibrated blend
(mean=0.4006, sd=0.0096): **-0.0217 improvement** -- ensembling (item 3)
and recalibration (items 2+4) compound rather than overlap. Pooled
candidate (mild optimism as always, see notebook 15's caveat): log
loss=0.3684, AUROC=0.9161, ECE=0.0369.

**Interpretation of the parameter shift**: T_cnn flipped from ~1.00
(notebook 15, rung3 alone) to 0.50 here -- the CNN side needs sharpening
now, whereas T_baseline flipped from ~0.50 to ~1.00. This matches the
literature-predicted mechanism exactly (Opus's strategic review,
`project_dat_parkinson_strategic_roadmap.md`): averaging more
(imperfectly-correlated) model outputs systematically produces
**under-confident** probabilities -- true for the 5-variant, 25-array
ensemble in a way it wasn't yet for the 5-array rung3-only ensemble (whose
own ECE, 0.0316, was already low). This isn't a coincidence or a
grid-search fluke; it's the calibration correcting for a known,
mechanistically-expected side effect of the exact ensembling decision made
in notebook 16.

**Decision: adopt T_cnn=0.50, T_baseline=1.00, w=0.45 as the final blend
recipe**, on top of the 5-variant (125-checkpoint) ensemble composition
from notebook 16. LOFO mean 0.3789 vs. the real leaderboard's 0.4648 is a
large, honestly-validated gap -- this is now the strongest candidate this
project has produced, and it should be treated as such: implement
carefully, verify with both smoke tests, before spending one of the 2
remaining real submissions on it. This is the last local-CV notebook in
the roadmap before touching `submission_src/main.py` for real -- see
`project_dat_parkinson_strategic_roadmap.md` for the packaging checklist
(sklearn pickle risk, inference batch size, checkpoint-list extension)
still outstanding before that submission.